<a href="https://colab.research.google.com/github/kvnxmars/cfg-vs-llm-mcq-synthesis/blob/main/LLM_Regex_Synthesiser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating the Computational Efficiency, Formal Correctness, and Pedagogical Realism of LLM-Synthesized Regular Expression Assessment Materials

This notebook implements the complete research pipeline for evaluating and comparing **Symbolic Context-Free Grammar (CFG) MCQ generation** against **Probabilistic Neural Generation (using the GPT-5.6 Luna model)**.

### Pipeline Architecture:
1. **Cell 1 & 2: Environment Setup and Dataset Ingestion:** Installs dependencies and parses the ground-truth benchmark questions from Ade-Ibijola's (2018) 10,000 MCQ dataset.
2. **Cell 3: Pedagogical MCQ Wrappers:** Configures the prompt layouts for **Figure 7 (Classical Single-Choice MCQ)** and **Figure 8 (Multi-Answer MCQ)**.
3. **Cell 4: Neural Synthesis & Custom Cost Logging:** Orchestrates parallel LLM generation runs using GPT-5.6 Luna, implementing token-level cost tracking (including caching) and latency metrics.
Cell 1 (Code)

In [6]:
#environment setup & libraries
print("Installing libraries..")

#install opneai for gpt4o prompting and fpdf2 for pdf generation
!pip install --quiet openai fpdf2 pypdf
print("Libraries successfully installed!")

print("Assembling imports..")
import os
import re
import sys
import time
import random
import pypdf
import json

from google.colab import files

print("Setup complete.. Ready to load pdf.")

Installing libraries..
Libraries successfully installed!
Assembling imports..
Setup complete.. Ready to load pdf.


## Cell 2: Ground-Truth Dataset Parsing and Text Normalization


This cell ingests both uploaded baseline PDFs and merges them into a single, unified database (`baseline_tasks.json`) of exactly **500 benchmark tasks**:
1. **Single-Answer Baseline:** Parses the first **250 tasks** from `10000 iterations of MCQs and Solutions.pdf` (Figure 7 layout).
2. **Multi-Answer Baseline:** Parses the first **250 tasks** from `1000 iterations of MCQs - multiple correct solutions.pdf` (Figure 8 layout).

### Normalization and Cleansing:
* **Asterisk Normalization:** Converts mathematical Unicode stars (\\(\ast\\) or `\u2217`) to standard keyboard stars (`*`) to ensure compatibility with our automata engine.
* **Alphabet Boundary Restricting:** Dynamically extracts the restricted alphabet (\\(\Sigma\\)) for each question, filtering out spaces and empty-string lambda symbols (\\(\lambda\\)).
* **Deferred Multi-Answer Keys:** For multi-answer tasks, `labeled_correct` is initialized to `null`. Cell 5 will programmatically grade and label these options using our local DFA verifier.






In [7]:
# =====================================================================
# CELL 2: UPLOAD & PARSE BOTH PDF BASKETS (SINGLE & MULTI) WITH PREVIEWS
# =====================================================================
import os
import re
import json
import pypdf
import sys
from google.colab import files

# 1. Enforce both files exist in the Google Colab workspace
required_files = [
    "10000 iterations of MCQs and Solutions.pdf",
    "1000 iterations of MCQs - multiple correct solutions.pdf"
]

for f_name in required_files:
    if not os.path.exists(f_name):
        print(f"Please upload '{f_name}':")
        uploaded = files.upload()
    else:
        print(f"Found existing '{f_name}' in workspace.")

def clean_regex_text(raw):
    """
    pypdf splits superscript exponents (∗, +, ?) onto their own line,
    e.g. 'g\\n∗\\nyi+' instead of 'g∗yi+'. Strip ALL whitespace/newlines
    and normalize the unicode asterisk to a plain '*'.
    """
    return re.sub(r"\s+", "", raw).replace("∗", "*")

# 2. Ingest Single-Answer PDF
def parse_single_answer_pdf(pdf_path, count=250):
    print(f"\n🔄 Parsing {count} tasks from Single-Answer PDF...")
    reader = pypdf.PdfReader(pdf_path)
    text = "\n".join(
    page.extract_text(extraction_mode="layout")
    for page in reader.pages
    if page.extract_text()
)

    # Normalize mathematical Unicode asterisks to standard regex stars
    text = text.replace("∗", "*")

    # Corrected regex (no double backslashes) matching: (1). Rex = ...
    pattern = re.compile(
        r"\((\d+)\)\.\s*Rex\s*=\s*(.*?);\s*Σ\s*=\s*\{([^}]+)\}\.\s*"
        r"(.*?)Answer:\s*([A-E])\.",
        re.DOTALL
    )

    opt_pattern = re.compile(r"([A-E])\.\s*(.*?)\s*(?=(?:[A-E]\.)|$)", re.DOTALL)

    tasks = []
    for match in pattern.finditer(text):
        task_id = int(match.group(1))
        target_regex = clean_regex_text(match.group(2))
        alphabet = [
            sym.strip()
            for sym in match.group(3).replace(" ", "").split(",")
            if sym.strip() and sym.strip() != "λ"
        ]
        answer_letter = match.group(5)
        option_block = match.group(4)

        letter_to_text = {}
        for letter, opt_text in opt_pattern.findall(option_block):
            letter_to_text[letter] = clean_regex_text(opt_text) if False else opt_text.strip()

        options = []
        for letter in ["A", "B", "C", "D", "E"]:
            options.append({
                "string": letter_to_text.get(letter, ""),
                "labeled_correct": (letter == answer_letter)
            })

        tasks.append({
            "task_id": task_id,
            "wrapper_type": "Figure 7 (Single-Choice)",
            "target_regex": target_regex,
            "alphabet": alphabet,
            "options": options
        })
        if len(tasks) >= count:
            break


    if not tasks:
        print("⚠️  0 matches — dumping a text sample so you can inspect the actual format:")
        print(repr(text[:1500]))

    print(f"✅ Extracted {len(tasks)} classical single-choice tasks.")
    return tasks



# 3. Ingest Multi-Answer PDF
def parse_multi_answer_pdf(pdf_path, count=250):
    print(f"\n🔄 Parsing {count} tasks from Multi-Answer PDF...")
    reader = pypdf.PdfReader(pdf_path)
    text = "".join([page.extract_text() + "\n" for page in reader.pages if page.extract_text()])



    # Corrected regex (no double backslashes) matching: (No. 1). Let r = ...
    pattern = re.compile(
        r"\(No\.\s*(\d+)\)\.\s*Let\s*r\s*=\s*(.*?)\.\s*"
        r"Which\s*of\s*the\s*following.*?\?\s*Take\s*"
        r"Σ\s*=\s*\{([^}]+)\}\.\s*"
        r"(.*?)Circle\s*your\s*answers\.",
        re.DOTALL
    )

    tasks = []
    for match in pattern.finditer(text):
        task_id = int(match.group(1)) + 10000  # offset to prevent collision with single-answer IDs
        target_regex = clean_regex_text(match.group(2))
        alphabet = [
            sym.strip()
            for sym in match.group(3).replace(" ", "").split(",")
            if sym.strip() and sym.strip() != "λ"
        ]
        raw_options = match.group(4)

        # Options are comma-separated on their own line(s), trailing comma before "Circle..."
        opts = [o.strip() for o in raw_options.strip().rstrip(",").split(",") if o.strip()]

        options = [{"string": opt, "labeled_correct": None} for opt in opts]

        tasks.append({
            "task_id": task_id,
            "wrapper_type": "Figure 8 (Multi-Answer)",
            "target_regex": target_regex,
            "alphabet": alphabet,
            "options": options
        })
        if len(tasks) >= count:
            break
    if not tasks:
        print("⚠️  0 matches — dumping a text sample so you can inspect the actual format:")
        print(repr(text[:1500]))

    print(f"✅ Extracted {len(tasks)} multi-answer tasks.")
    return tasks

# 4. Execute parser and save output
single_tasks = parse_single_answer_pdf("10000 iterations of MCQs and Solutions.pdf", 250)
multi_tasks = parse_multi_answer_pdf("1000 iterations of MCQs - multiple correct solutions.pdf", 250)
unified_dataset = single_tasks + multi_tasks

with open("baseline_tasks.json", "w") as f:
    json.dump(unified_dataset, f, indent=4)

print(f"\n💾 Saved {len(unified_dataset)} unified ground-truth tasks to 'baseline_tasks.json'!")

# 5. Output exact JSON previews for validation
if len(single_tasks) > 0:
    print("\n ---Sample: Single-Answer Task ---")
    print(json.dumps(single_tasks, indent=2))

if len(multi_tasks) > 0:
    print("\n ---Sample: Multi-Answer Task ---")
    print(json.dumps(multi_tasks, indent=2))

Streaming output truncated to the last 5000 lines.
        "string": "bbbxggmmm",
        "labeled_correct": null
      },
      {
        "string": "gmm",
        "labeled_correct": null
      },
      {
        "string": "ubbbbgg",
        "labeled_correct": null
      },
      {
        "string": "ubbbbggm",
        "labeled_correct": null
      },
      {
        "string": "lg",
        "labeled_correct": null
      }
    ]
  },
  {
    "task_id": 10101,
    "wrapper_type": "Figure 8 (Multi-Answer)",
    "target_regex": "k+b+",
    "alphabet": [
      "l",
      "j",
      "k",
      "b"
    ],
    "options": [
      {
        "string": "kbbbb",
        "labeled_correct": null
      },
      {
        "string": "kkkkb",
        "labeled_correct": null
      },
      {
        "string": "kkkkbb",
        "labeled_correct": null
      },
      {
        "string": "llll j j j jkk",
        "labeled_correct": null
      },
      {
        "string": "lllk",
        "labeled_correct": nu

## Cell 3: Pedagogical MCQ Wrappers &amp; Strict Operator Constraints

This cell configures the **System Prompt** and **User Prompt Templates** for Figure 7 and Figure 8 MCQ wrappers.
### FLA Operator Constraints:
To isolate the neural model's logical reasoning engine rather than its memory of programming-specific regex shorthand (such as \`\\d\`, \`\\w\`, or anchors like \`^\` and \`$\`), the system prompt strictly constrains the LLM to use only core theoretical operators: \* Concatenation (implied) \* Alternation / Union (\`|\`) \* Kleene Star (\`\*\`) \* Kleene Plus (\`+\`) \* Optionality (\`?\`) \* Explicit grouping parentheses (\`()\`)

In [3]:
# =====================================================================
# CELL 3: MCQ WRAPPERS & SYSTEM PROMPTS (FIGURE 7 & FIGURE 8)
# =====================================================================

SYSTEM_PROMPT = """
You are an expert tutor in Formal Languages and Automata (FLA) theory. Your task is to generate a pedagogically sound Multiple-Choice Question (MCQ) item for an undergraduate Computer Science practice exam.

You must strictly adhere to the following language and syntax constraints:
1. ALPHABET CONSTRAINT (Σ): You may ONLY use the characters explicitly provided in the user's alphabet. No other characters or shorthand symbols (such as \\d, \\w, \\s) are allowed.
2. OPERATOR CONSTRAINT: You may ONLY use the core theoretical operators of regular expressions:
   - Concatenation (implied, e.g., ab)
   - Alternation / Union (using "|" or "+", e.g., a|b)
   - Kleene Star ("*")
   - Kleene Plus ("+")
   - Optionality ("?")
   - Explicit grouping parentheses ("( )")
   Do NOT use lookarounds, backreferences, anchors (^ or $), or any PCRE programming extensions.

You must respond with STRICT JSON ONLY. Do not wrap the JSON in markdown code fences, and do not include any conversational preamble or postscript.

JSON SCHEMA:
{
  "synthesized_regex": "<the regular expression r_LLM you constructed>",
  "shuffled_options": [
    {"label": "A", "string": "<string_1>", "labeled_correct": true},
    {"label": "B", "string": "<string_2>", "labeled_correct": false},
    {"label": "C", "string": "<string_3>", "labeled_correct": false},
    {"label": "D", "string": "<string_4>", "labeled_correct": false},
    {"label": "E", "string": "<string_5>", "labeled_correct": false}
  ],
  "pedagogical_rationales": {
    "regex_design": "Explain in 1 sentence how you constructed the regex.",
    "option_mapping": {
       "A": "Why this string matches/does not match.",
       "B": "Why this string matches/does not match.",
       "C": "Why this string matches/does not match.",
       "D": "Why this string matches/does not match.",
       "E": "Why this string matches/does not match."
    }
  }
}
"""

# Wrapper 1: Figure 7 Classic Format
WRAPPER_FIG7_TEMPLATE = """
Target Language Specification: {description}
Alphabet (Σ): {{{alphabet_symbols}}}

Generate one classical MCQ item following this exact problem wrapper layout from Figure 7:

"For each of the following regular expressions, defined over the indicated alphabet, determine (from options A to E) which string is accepted by the expression.
Let r = {target_regex}. Take Σ = {{{alphabet_symbols}}}.

[List five randomly shuffled strings here; four wrong and one correct answer.]"

Instructions:
1. Synthesize the regular expression (it must be equivalent to {target_regex}).
2. Generate exactly FIVE options (labeled A to E).
3. Exactly ONE option must be mathematically correct (labeled_correct = true), and the other FOUR must be incorrect distractors.

Provide your output strictly matching the required JSON schema.
"""

# Wrapper 2: Figure 8 Multi-Answer Format
WRAPPER_FIG8_TEMPLATE = """
Target Language Specification: {description}
Alphabet (Σ): {{{alphabet_symbols}}}
Number of Correct Options (m): {num_correct}

Generate one multi-answer MCQ item following this exact problem wrapper layout from Figure 8:

"Let r = [synthesized_regex]. Which of the following strings is in L(r)? Take Σ = {{{alphabet_symbols}}}.
[List 5 randomly shuffled strings here; {num_correct} correct strings, and {num_wrong} wrong strings.]"

Instructions:
1. Synthesize the regular expression (it must be equivalent to {target_regex}).
2. Generate exactly FIVE options (labeled A to E).
3. Exactly {num_correct} options must be mathematically correct (labeled_correct = true), and the remaining {num_wrong} options must be incorrect distractors (where {num_correct} must be between 1 and 4).

Provide your output strictly matching the required JSON schema.
"""

print("✅ Step 3 complete! Prompt templates for Figure 7 and Figure 8 are compiled.")

✅ Step 3 complete! Prompt templates for Figure 7 and Figure 8 are compiled.




## Cell 4A: Free Llama-3.1 Synthesis Pipeline (Groq Engine)

This cell executes the 500-task benchmark using **Meta's Llama-3.1-70B** hosted on Groq's high-speed LPU infrastructure.

### Features:
- **Zero-Cost Execution:** Runs entirely on Groq's free developer tier (API key from console.groq.com).
- **High Throughput:** Executes at ~250 tokens/second without artificial daily rate limit caps.
- **Dedicated Output:** Saves results to `cumulative_tasks_llama.json` to preserve data for comparative model analysis against OpenAI.






In [5]:
# =====================================================================
# CELL 4A: FREE GROQ (LLAMA / GPT-OSS) TEST-SET / SYNTHESIS LOOP
# =====================================================================
import os, sys, time, json, random
from openai import OpenAI
from getpass import getpass

# -- 0. Mount Drive so incremental progress survives a disconnect --
# Colab's local disk (/content/...) is fine WITHIN a session — that's
# how your earlier runs saved cumulative_tasks_llama.json successfully.
# But if the runtime disconnects (idle timeout, ~90min job), that disk
# is wiped. Mount Drive once and write there instead.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/regex_llm_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- 1. Groq Provider Configuration --
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "openai/gpt-oss-120b"   # <-- STRING, not a list

GROQ_API_KEY = getpass("Enter Groq API key (console.groq.com): ")
groq_client = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)

# -- 2. TEST-SET LIMIT --
# None = full 500-task run. Set to 5 or 100 to eyeball output before committing.
MAX_TASKS = 5

# -- 3. Ingest baseline dataset --
baseline_path = os.path.join(OUTPUT_DIR, "baseline_tasks.json")
if not os.path.exists(baseline_path) and os.path.exists("baseline_tasks.json"):
    baseline_path = "baseline_tasks.json"   # fallback if Cell 2 hasn't been pointed at Drive yet

try:
    with open(baseline_path, "r") as f:
        all_baseline_tasks = json.load(f)
    print(f"Loaded {len(all_baseline_tasks)} benchmark tasks from '{baseline_path}'.")
except FileNotFoundError:
    print("Error! baseline_tasks.json missing. Run Cell 2 first.")
    sys.exit(1)

# -- 4. Resume log: ONLY successful tasks are ever written here --
llama_output_file = os.path.join(OUTPUT_DIR, "cumulative_tasks_llama.json")
llama_dataset = []
if os.path.exists(llama_output_file):
    with open(llama_output_file, "r") as f:
        llama_dataset = json.load(f)
    print(f"Resuming: {len(llama_dataset)} tasks already completed.")
existing_ids = {item["task_id"] for item in llama_dataset}

# -- 5. Execution loop --
print(f"\nLaunching {GROQ_MODEL} (Groq) Synthesis Loop..")
print("----------------------------------------------")

processed_this_run = 0
for idx, task in enumerate(all_baseline_tasks):
    if MAX_TASKS is not None and processed_this_run >= MAX_TASKS:
        print(f"\nHit MAX_TASKS={MAX_TASKS}. Stopping for review.")
        break

    task_id = task["task_id"]
    if task_id in existing_ids:
        continue

    target_regex = task["target_regex"]
    alphabet_list = task["alphabet"]
    alphabet_str = ", ".join(alphabet_list)
    description = f"strings matching the regular expression {target_regex}"
    wrapper_type = task["wrapper_type"]

    if wrapper_type == "Figure 7 (Single-Choice)":
        user_prompt = WRAPPER_FIG7_TEMPLATE.format(
            description=description, target_regex=target_regex, alphabet_symbols=alphabet_str
        )
    else:
        rng = random.Random(task_id)          # seeded -> identical prompt every run/model
        num_correct = rng.randint(2, 4)
        user_prompt = WRAPPER_FIG8_TEMPLATE.format(
            description=description, target_regex=target_regex, alphabet_symbols=alphabet_str,
            num_correct=num_correct, num_wrong=5 - num_correct
        )

    print(f"[{GROQ_MODEL}] Task {idx + 1}/{len(all_baseline_tasks)} [ID: {task_id}] | {wrapper_type}...")

    parsed_response = None
    prompt_tokens = completion_tokens = 0
    parse_error = None

    for attempt in range(4):
        t0 = time.perf_counter()
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            duration = time.perf_counter() - t0
            prompt_tokens = response.usage.prompt_tokens
            completion_tokens = response.usage.completion_tokens
            raw_content = response.choices[0].message.content.strip()   # <-- [0] added
            parsed_response = json.loads(raw_content)
            parse_error = None
            break
        except Exception as e:
            duration = time.perf_counter() - t0
            parse_error = str(e)
            if attempt < 3:
                wait = 2 * (2 ** attempt)
                print(f"  attempt {attempt+1} failed ({parse_error[:80]}); retry in {wait}s")
                time.sleep(wait)

    if parsed_response is None:
        print(f"  GIVING UP on task {task_id}: {parse_error}")
        continue   # NOT added to existing_ids -> will retry next run

    record = {
        "task_id": task_id,
        "model_engine": GROQ_MODEL,
        "wrapper_type": wrapper_type,
        "target_regex": target_regex,
        "alphabet": alphabet_list,
        "synthesized_regex": parsed_response.get("synthesized_regex"),
        "options": parsed_response.get("shuffled_options"),
        "rationales": parsed_response.get("pedagogical_rationales"),
        "metrics": {
            "duration_s": duration,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "parse_error": None,
        }
    }

    llama_dataset.append(record)
    existing_ids.add(task_id)
    processed_this_run += 1

    with open(llama_output_file, "w") as f:
        json.dump(llama_dataset, f, indent=4)

    print(f"  {duration:.2f}s | in: {prompt_tokens} | out: {completion_tokens}")
    time.sleep(0.5)
    print("-------------------------------------------------------------")

print(f"\nRun complete! {len(llama_dataset)} tasks saved to '{llama_output_file}'.")

Importing important things..
Enter Groq API key (console.groq.com): ··········
Loaded 500 benchmark tasks.

=== openai/gpt-oss-120b -> cumulative_tasks_openai_gpt-oss-120b.json ===
  json_object mode unavailable (Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}); falling back to fence-stripping.
[openai/gpt-oss-120b] 1/500 [ID: 1] | Figure 7 (Single-Choice)
  3.93s | in: 730 | out: 1772
[openai/gpt-oss-120b] 2/500 [ID: 2] | Figure 7 (Single-Choice)
  2.95s | in: 734 | out: 1126
[openai/gpt-oss-120b] 3/500 [ID: 3] | Figure 7 (Single-Choice)
  2.28s | in: 734 | out: 927
[openai/gpt-oss-120b] 4/500 [ID: 4] | Figure 7 (Single-Choice)
  2.59s | in: 733 | out: 1055
[openai/gpt-oss-120b] 5/500 [ID: 5] | Figure 7 (Single-Choice)
  2.43s | in: 745 | out: 1029
[openai/gpt-oss-120b] 6/500 [ID: 6] | Figure 7 (Single-Choice)
  18.35s | in: 741 | out: 1467
[openai/gpt-oss

KeyboardInterrupt: 